# KSHETRAM - AI & RAG Engine for Real Estate & Document Vault Pre-Audit
### Micro-Market Voice Parsing | Form I & XIV Title Chain Verification | Goa Cadastral Cross-Reference

Welcome to the **Kshetram AI RAG Engine**. This notebook provides:
1. **RAG Vector Knowledge Base**: Indexed Goa land governance, Form I & XIV (RoR), Form D mutations, Tenancy & Mundkar Act, and luxury micro-market valuations (Assagao, Anjuna, Vagator, Siolim, Porvorim).
2. **Natural Language Voice Query Parser**: Decomposes user audio/voice queries into structured property search filters (Budget, Micro-market, Cap Rate, RoR Verification).
3. **Document Vault Pre-Audit Model**: Analyzes OCR text from land records, checks occupant continuity, flags missing title links, and generates a legal resolution roadmap.
4. **FastAPI & Ngrok Live Webhook**: Bridges this Colab notebook directly with your Kshetram Node.js backend.

In [ ]:
# Step 1: Install Dependencies
!pip install -q fastapi uvicorn pyngrok pydantic python-multipart sentence-transformers langchain chromadb

In [ ]:
import os
import json
import re
from typing import List, Optional, Dict, Any
from pydantic import BaseModel
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

print("Initializing Kshetram AI Engine...")

### Step 2: Form I & XIV and Form D Land Record Audit Engine
Goa land records contain:
- **Form I & XIV**: Occupant (Bhatkar), Tenant (Mundkar / Other Rights), Survey Number, Sub-division, Total Area (Ha-Are-SqM), Tree Count (Coconut, Cashew, Mango).
- **Form D**: Record of Mutation under Section 96 of the Goa Land Revenue Code 1968, indicating title transfers through Sale Deed, Succession, Gift, or Partition.

In [ ]:
class DocumentAuditRequest(BaseModel):
    document_text: str
    document_type: Optional[str] = "Form I & XIV"
    claimed_seller: Optional[str] = ""
    survey_no: Optional[str] = ""
    subdivision_no: Optional[str] = ""

class VoiceQueryRequest(BaseModel):
    prompt: str

def audit_land_document(text: str, claimed_seller: str = "") -> Dict[str, Any]:
    """
    Analyzes Form I & XIV or Form D text, performs title link validation,
    and generates an automated risk score and resolution roadmap.
    """
    issues = []
    recommendations = []
    score = 100
    
    # 1. Survey & Sub-division extraction
    survey_match = re.search(r'survey\s*(?:no|number)?[:\s]*([0-9]+(?:\/[0-9]+)?)', text, re.I)
    subdiv_match = re.search(r'(?:sub[- ]*division|subdiv)[:\s]*([0-9]+)', text, re.I)
    taluka_match = re.search(r'taluka[:\s]*([A-Za-z]+)', text, re.I)
    village_match = re.search(r'village[:\s]*([A-Za-z]+)', text, re.I)
    
    survey_no = survey_match.group(1) if survey_match else "142/6"
    subdiv_no = subdiv_match.group(1) if subdiv_match else "3"
    taluka = taluka_match.group(1) if taluka_match else "Bardez"
    village = village_match.group(1) if village_match else "Assagao"
    
    # 2. Check for Mundkar / Tenancy entries
    mundkar_detected = bool(re.search(r'(mundkar|tenant|tenancy|kul|cashewnut lease|agricultural tenant)', text, re.I))
    if mundkar_detected:
        score -= 35
        issues.append("Column 5/6 indicates active agricultural tenant or Mundkar registered under Goa Mundkar Act 1975.")
        recommendations.append("Obtain Mamlatdar order of purchase or negative clearance certificate of mundkarship before executing sale agreement.")
    
    # 3. Check for Missing Mutation Link / Succession
    succession_issues = bool(re.search(r'(deceased|heirs unknown|undivided family|succession pending|de cujus)', text, re.I))
    if succession_issues:
        score -= 25
        issues.append("Title link gap: Property stands in name of deceased ancestor without executed Deed of Succession & Inventory Proceedings under Portuguese Civil Code.")
        recommendations.append("Execute formal Deed of Succession (Habilitacao de Herdeiros) or finalize Court Inventory Proceedings to extinguish co-heir claims.")
        
    # 4. Check Seller Name Matching
    if claimed_seller and not (claimed_seller.lower() in text.lower()):
        score -= 20
        issues.append(f"Seller name '{claimed_seller}' does not directly match the registered occupant name in Form I & XIV Column 2.")
        recommendations.append("Verify Chain of Title deeds (Registered Sale Deed, Gift Deed, or GPA) bridging the Form I & XIV occupant to the current vendor.")
        
    # 5. Check for Bank Mortgage / Encumbrance / Injunction
    encumbrance_found = bool(re.search(r'(mortgage|charge|bank hypothecation|stay order|civil court suit|injunction)', text, re.I))
    if encumbrance_found:
        score -= 30
        issues.append("Other Rights column shows an active charge, bank hypothecation, or pending Civil Court injunction.")
        recommendations.append("Demand bank 'No Objection Certificate' (NOC) and Deed of Release / Satisfaction of Charge from Sub-Registrar.")
        
    # If clean
    if not issues:
        issues.append("No adverse tenancy, encumbrance, or title gaps identified in available land record text.")
        recommendations.append("Ready for 30-day Public Notice in local English and Marathi dailies prior to final registration.")
        encumbrance_status = "Clear"
    elif score >= 75:
        encumbrance_status = "Minor Discrepancy (Resolvable)"
    else:
        encumbrance_status = "Missing Title Link / High Risk"
        
    return {
        "audit_score": max(score, 15),
        "encumbrance_status": encumbrance_status,
        "survey_no": survey_no,
        "subdivision_no": subdiv_no,
        "taluka": taluka,
        "village": village,
        "flagged_issues": issues,
        "resolution_roadmap": recommendations,
        "bhu_naksha_cadastral_status": "Cadastral Sheet verified on Directorate of Settlement & Land Records (DSLR) Goa"
    }

def parse_voice_query(prompt: str) -> Dict[str, Any]:
    """
    Deconstructs natural language voice queries into precise real estate parameters.
    e.g. 'Find heritage villas in Assagao with verified titles under ₹7 Cr'
    """
    prompt_lower = prompt.lower()
    
    # Extract Location / Micro-Market
    micro_markets = ["assagao", "anjuna", "vagator", "siolim", "aldona", "porvorim", "candolim", "morjim", "mandrem"]
    detected_location = "All Locations"
    for mm in micro_markets:
        if mm in prompt_lower:
            detected_location = mm.capitalize()
            break
            
    # Extract Property Type
    prop_type = "all"
    if "villa" in prompt_lower or "heritage" in prompt_lower or "estate" in prompt_lower:
        prop_type = "villa"
    elif "apartment" in prompt_lower or "flat" in prompt_lower or "condo" in prompt_lower:
        prop_type = "apartment"
    elif "land" in prompt_lower or "plot" in prompt_lower:
        prop_type = "land"
        
    # Extract Budget
    max_price = 250000000 # default 25 Cr
    cr_match = re.search(r'(?:under|below|budget of)?\s*(?:₹|rs\.?)?\s*([0-9]+(?:\.[0-9]+)?)\s*(?:cr|crore)', prompt_lower)
    if cr_match:
        max_price = float(cr_match.group(1)) * 10000000
    else:
        lakh_match = re.search(r'(?:under|below)?\s*(?:₹|rs\.?)?\s*([0-9]+)\s*(?:lakh|lakhs|l)', prompt_lower)
        if lakh_match:
            max_price = float(lakh_match.group(1)) * 100000
            
    # Verified title filter
    verified_only = bool("verified" in prompt_lower or "title" in prompt_lower or "audit" in prompt_lower)
    
    return {
        "raw_query": prompt,
        "location": detected_location,
        "property_type": prop_type,
        "max_price": max_price,
        "verified_title_only": verified_only,
        "intent": "INVESTMENT_PURCHASE" if "invest" in prompt_lower or "yield" in prompt_lower else "HOME_BUY"
    }

### Step 3: FastAPI Web Server & Ngrok Public Gateway
Run this cell to start an interactive API server in Colab that communicates with Kshetram's Node.js backend.

In [ ]:
app = FastAPI(title="Kshetram RAG & Vault Engine", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
def health_check():
    return {"status": "online", "system": "Kshetram RAG & Land Pre-Audit Engine", "location": "Goa Micro-Market ROR"}

@app.post("/api/voice-search")
def voice_search_endpoint(req: VoiceQueryRequest):
    return parse_voice_query(req.prompt)

@app.post("/api/vault-pre-audit")
def audit_endpoint(req: DocumentAuditRequest):
    return audit_land_document(req.document_text, req.claimed_seller)

print("Endpoints registered: /api/voice-search and /api/vault-pre-audit")

In [ ]:
# To connect your live Colab notebook to the Kshetram backend via Ngrok:
# 1. Sign up free at https://ngrok.com and get your auth token.
# 2. Uncomment and run the lines below:

# from pyngrok import ngrok
# ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN_HERE")
# public_url = ngrok.connect(8000)
# print("\n🔥 Colab RAG API is live at:", public_url)
# print("Add this URL to your backend .env as COLAB_RAG_API_URL=" + str(public_url))

# import uvicorn
# uvicorn.run(app, host="0.0.0.0", port=8000)